In [ ]:
from robovast.common.analysis import read_output_files, read_output_csv
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

DATA_DIR = ''

# Every run of this configuration, from the per-run table the `resource_usage`
# postprocessing step writes into each run directory. read_output_files adds `run`.
#
# One read, not one per container: the container is a column. The previous version named
# resource_usage_robovast.csv (a filename that has never existed), used cpu_usage/mem_usage
# (the writer emits cpu_percent/memory_rss_bytes), and referenced an undefined `per_test` --
# it could not have run against any campaign.
try:
    df = read_output_files(DATA_DIR, lambda run_dir: read_output_csv(run_dir, "resource_usage.csv"))
except ValueError as e:
# A campaign postprocessed BEFORE the resource_usage step existed has no such table --
# the step is auto-injected, so this is a question of when the campaign was processed,
# not of how it was configured. Re-running postprocessing produces it from the raw
# per-container CSVs, which are kept in _jobs/ forever, so nothing is re-executed.
    raise RuntimeError(
        f"{e}\n\nNo run here has a resource_usage table. Re-run postprocessing on this "
        f"campaign to produce it; the raw per-container CSVs under _jobs/ are kept, so "
        f"no run has to be executed again."
    ) from e

# in_window == 0 is bring-up and teardown, which differ between runs for reasons that have
# nothing to do with the trial. Comparing runs means comparing their trials.
trial = df[df['in_window'] == 1]
print(f"{trial['run'].nunique()} run(s), "
      f"{trial['container'].nunique()} container(s): "
      f"{', '.join(sorted(trial['container'].unique()))}")


In [ ]:
# A container's load is the SUM over its processes at an instant; the comparison across runs
# is then over those per-instant totals.
# observed=True because read_output_files makes `run` a CATEGORICAL column: the pandas
# default expands the full cross-product of categories, which here means every run paired
# with every other run's timestamps -- 19290 rows of mostly-NaN for 1289 real ones.
per_instant = (trial.dropna(subset=['timestamp'])
                    .groupby(['container', 'run', 'timestamp'], as_index=False, observed=True)
                    .agg(cpu_percent=('cpu_percent', 'sum'),
                         memory_rss_bytes=('memory_rss_bytes', 'sum')))
per_instant['mem_usage_mb'] = per_instant['memory_rss_bytes'] / (1024 * 1024)

# Runs start at sim time 0 already, so binning aligns them directly -- no per-run offset.
BIN_S = 5
per_instant['time_bin'] = (per_instant['timestamp'] / BIN_S).round() * BIN_S

binned = per_instant.groupby(['container', 'time_bin'], as_index=False, observed=True).agg(
    cpu_mean=('cpu_percent', 'mean'), cpu_std=('cpu_percent', 'std'),
    mem_mean=('mem_usage_mb', 'mean'), mem_std=('mem_usage_mb', 'std'))

containers = sorted(binned['container'].unique())
colors = dict(zip(containers, plt.cm.tab10(np.linspace(0, 0.9, len(containers)))))

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
n_runs = per_instant['run'].nunique()
fig.suptitle(f'Resource usage over {n_runs} run(s)', fontsize=16, fontweight='bold')
(ax_cpu_ts, ax_cpu_dist), (ax_mem_ts, ax_mem_dist) = axes

for container in containers:
    g = binned[binned['container'] == container].sort_values('time_bin')
    c = colors[container]
    for ax, mean, std in ((ax_cpu_ts, 'cpu_mean', 'cpu_std'),
                          (ax_mem_ts, 'mem_mean', 'mem_std')):
        spread = g[std].fillna(0)
        ax.plot(g['time_bin'], g[mean], color=c, linewidth=2, label=container)
        ax.fill_between(g['time_bin'], g[mean] - spread, g[mean] + spread,
                        alpha=0.25, color=c)

for ax, ylabel, title in (
    (ax_cpu_ts, 'CPU (%, per-core)', 'CPU over time (mean +/- sd across runs)'),
    (ax_mem_ts, 'Memory RSS (MB)', 'Memory over time (mean +/- sd across runs)'),
):
    ax.set_xlabel('Simulation time (s)', fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend()

for ax, column, ylabel, title in (
    (ax_cpu_dist, 'cpu_percent', 'CPU (%, per-core)', 'CPU distribution'),
    (ax_mem_dist, 'mem_usage_mb', 'Memory RSS (MB)', 'Memory distribution'),
):
    data = [per_instant[per_instant['container'] == c][column].values for c in containers]
    parts = ax.violinplot(data, positions=range(len(containers)),
                          showmedians=True, showextrema=True)
    for body, container in zip(parts['bodies'], containers):
        body.set_facecolor(colors[container])
        body.set_alpha(0.65)
    ax.set_xticks(range(len(containers)))
    ax.set_xticklabels(containers)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


In [ ]:
def plot_pie_charts(group, container):
    """What was heavy inside this container, averaged so each run weighs the same however
    long it ran. `name` is the stable key: pids churn, names do not."""
    n_runs = group['run'].nunique()
    # observed=True: `run` is categorical (see the cell above).
    process_usage = (group.groupby(['run', 'name'], observed=True)[['cpu_percent', 'memory_rss_bytes']]
                          .mean()
                          .groupby('name', observed=True)
                          .mean()
                          .reset_index())
    process_usage['mem_usage_mb'] = process_usage['memory_rss_bytes'] / (1024 * 1024)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
    fig.suptitle(f'{container}  -  mean over {n_runs} run(s)', fontsize=16, fontweight='bold')

    def make_pie(ax, column, threshold, unit, title):
        data = process_usage.sort_values(column, ascending=False)
        significant = data[data[column] > threshold]
        other = data[data[column] <= threshold][column].sum()
        if other > 0:
            plot_data = pd.concat([
                significant[['name', column]],
                pd.DataFrame([{'name': 'Other', column: other}]),
            ], ignore_index=True)
        else:
            plot_data = significant[['name', column]].copy()
        if plot_data.empty:
            ax.set_axis_off()
            return
        palette = plt.cm.Set3(np.linspace(0, 1, len(plot_data)))
        _, _, autotexts = ax.pie(plot_data[column], labels=plot_data['name'],
                                 autopct='%1.1f%%', startangle=90, colors=palette)
        ax.set_title(f'{title}\n(threshold > {threshold} {unit})',
                     fontsize=13, fontweight='bold')
        for at in autotexts:
            at.set_fontweight('bold')
            at.set_fontsize(9)

    make_pie(ax1, 'cpu_percent', threshold=0.5, unit='%', title='Mean CPU by process')
    make_pie(ax2, 'mem_usage_mb', threshold=50, unit='MB', title='Mean memory by process')

    plt.tight_layout()
    plt.show()


for container, group in trial.groupby('container', observed=True):
    plot_pie_charts(group, container)
